# Opdracht 1: Voorspel herseninfarct

**Teamnaam: 6** 

Aroetin Avanesi

Moreo Rahiembaks

Kaylim Taai

**Kaggle-gebruikersnamen:** _(nog invullen, verplicht volgens de opdracht)_

## Inhoudsopgave
1. Inleiding
2. Exploratieve Data Analyse
3. Evaluatie
4. Modellen
5. Conclusie en advies
6. Referenties

## 1. Inleiding

MedPredict is een medisch technologiebedrijf dat zorginstellingen en ziekenhuizen helpt om gezondheidsrisico's vroeg te herkennen. Het bedrijf heeft geanonimiseerde patiëntgegevens met demografische kenmerken, leefstijl en medische informatie.

Ongeveer 90% van de herseninfarcten hangt samen met risicofactoren die vooraf te meten en te beïnvloeden zijn, zoals hoge bloeddruk, roken en diabetes (O'Donnell et al., 2016). Als je patiënten met een verhoogd risico op tijd kunt herkennen, zijn artsen in staat om eerder in te grijpen met preventieve maatregelen.

Ons team bestaande uit junior data scientists is gevraagd om een machine learning model te bouwen dat dit risico voorspelt. MedPredict vergelijkt de oplossingen van verschillende teams via een interne Kaggle competitie.

### 1.1 Probleemstelling

MedPredict wil een model dat:
* op basis van patiëntgegevens voorspelt of iemand een herseninfarct kan krijgen
* zo min mogelijk patiënten met een verhoogd risico mist, maar ook zo min mogelijk valse alarmen geeft
* betrouwbaar en duidelijk genoeg is om artsen te ondersteunen

Omdat we hier te maken hebben met een medische toepassing, kunnen fouten leiden tot echte gevolgen. Een gemiste patiënt zou bijvoorbeeld geen preventie krijgen, terwijl een vals alarm iemand onnodige zorgen en lasten kan geven. Daarom kijken we niet alleen naar de score, maar ook naar de keuzes die we maken en beoordelen we hoe betrouwbaar het model is.

## 2. Exploratieve Data Analyse

### 2.1 Data inladen

In [10]:
from pathlib import Path

import pandas as pd


class StrokeData:
    """Laadt de datasets van de Kaggle competitie Predicting stroke."""

    def __init__(self, data_dir="data"):
        # Map waarin de csv bestanden staan
        self.data_dir = Path(data_dir)
        self.train = None
        self.test = None
        self.submission = None

    def load(self):
        """Leest alle drie de csv bestanden in als DataFrames."""
        self.train = pd.read_csv(self.data_dir / "train.csv")
        self.test = pd.read_csv(self.data_dir / "test.csv")
        self.submission = pd.read_csv(
            self.data_dir / "sample_submission.csv"
        )
        return self

    def check_shapes(self):
        """Toont het aantal rijen en kolommen van iedere dataset."""
        print(f"train:      {self.train.shape}")
        print(f"test:       {self.test.shape}")
        print(f"submission: {self.submission.shape}")


data = StrokeData().load()
data.check_shapes()
data.train.head()

train:      (33550, 22)
test:       (8388, 21)
submission: (8388, 2)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,52709,30.0,0,0,63.60,33.3,False,True,False,True,...,False,True,False,False,True,False,False,False,True,0
1,72295,75.0,1,0,215.17,48.0,True,False,False,False,...,False,False,True,False,True,False,False,True,False,0
2,26451,15.0,0,0,135.22,19.0,False,True,False,True,...,False,True,False,False,True,False,False,True,False,0
3,65210,47.0,0,0,64.89,28.2,False,True,False,False,...,False,False,True,False,False,True,False,True,False,0
4,69299,49.0,0,0,222.34,28.8,False,True,False,False,...,False,False,True,False,True,False,True,False,False,0


### 2.2 Uitleg kolommen

Iedere rij in de data is één patiënt. De kolommen zijn op Kaggle in het Engels beschreven; hieronder staat per kolom wat hij betekent.

| Kolom | Type | Uitleg |
|---|---|---|
| `id` | geheel getal | Uniek nummer van de patiënt. Zegt niets over de gezondheid, alleen nodig om de voorspellingen aan de juiste persoon te koppelen bij de Kaggle-inlevering. |
| `gender` | categorisch | Geslacht van de patiënt: `Male` (man), `Female` (vrouw) of `Other` (anders). |
| `age` | numeriek | Leeftijd van de patiënt in jaren. Bij baby's kan dit een kommagetal onder de 1 zijn. |
| `hypertension` | binair (0/1) | 1 = de patiënt heeft hoge bloeddruk (hypertensie), 0 = niet. |
| `heart_disease` | binair (0/1) | 1 = de patiënt heeft een hartziekte, 0 = niet. |
| `ever_married` | categorisch | Of de patiënt ooit getrouwd is geweest: `Yes` of `No`. |
| `work_type` | categorisch | Soort werk: `children` (kind, werkt nog niet), `Govt_job` (overheid), `Never_worked` (nooit gewerkt), `Private` (private sector) of `Self-employed` (zzp/eigen bedrijf). |
| `Residence_type` | categorisch | Woonomgeving: `Rural` (platteland) of `Urban` (stad). |
| `avg_glucose_level` | numeriek | Gemiddeld suikergehalte (glucose) in het bloed, in mg/dL. Een hoge waarde kan wijzen op diabetes. |
| `bmi` | numeriek | Body Mass Index: gewicht (kg) gedeeld door lengte² (m²). Rond 18,5 tot 25 geldt als gezond, boven 30 als obesitas. |
| `smoking_status` | categorisch | Rookgedrag: `formerly smoked` (vroeger gerookt), `never smoked` (nooit gerookt), `smokes` (rookt) of `Unknown` (onbekend). |
| `stroke` | binair (0/1) | **Doelvariabele.** 1 = de patiënt heeft een herseninfarct gehad, 0 = niet. Staat alleen in `train.csv`; deze kolom moeten we voorspellen voor `test.csv`. |

`stroke` is de afhankelijke variabele (y). Alle andere kolommen behalve `id` zijn onafhankelijke variabelen (features, X).

### 2.3 Datatypes en basisstatistieken

We maken een class `StrokeEDA` die alle verkennende analyses op de trainingsdata bundelt. Bij het aanmaken deelt de class de kolommen automatisch in drie groepen in:

* **numeriek**: getallen met veel verschillende waarden (`age`, `avg_glucose_level`, `bmi`);
* **binair**: getallen met alleen 0 en 1 (`hypertension`, `heart_disease`);
* **categorisch**: tekstkolommen (`gender`, `ever_married`, `work_type`, `Residence_type`, `smoking_status`).

Die indeling gebruiken we steeds om per soort kolom de juiste statistiek of grafiek te kiezen.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Eén vaste stijl voor alle grafieken in het notebook
sns.set_theme(style="whitegrid", palette="deep")


class StrokeEDA:
    """Verkennende data-analyse (EDA) op de trainingsdata van stroke."""

    def __init__(self, df, target="stroke", id_col="id"):
        self.df = df
        self.target = target
        self.id_col = id_col

        # Alleen de features, dus zonder id en doelvariabele
        features = df.drop(columns=[id_col, target], errors="ignore")
        number_cols = features.select_dtypes(include="number").columns

        # Getallen met maximaal twee waarden zijn binair (0/1)
        self.binary_cols = [
            col for col in number_cols if features[col].nunique() <= 2
        ]
        self.numeric_cols = [
            col for col in number_cols if col not in self.binary_cols
        ]
        self.categorical_cols = list(
            features.select_dtypes(exclude="number").columns
        )

    def overview(self):
        """Datatype, aantal unieke en ontbrekende waarden per kolom."""
        return pd.DataFrame({
            "datatype": self.df.dtypes.astype(str),
            "uniek": self.df.nunique(),
            "ontbrekend": self.df.isna().sum(),
            "ontbrekend_%": (self.df.isna().mean() * 100).round(2),
            "voorbeeld": self.df.iloc[0],
        })

    def numeric_stats(self):
        """Basisstatistieken van de numerieke en binaire kolommen."""
        cols = self.numeric_cols + self.binary_cols + [self.target]
        stats = self.df[cols].describe().T
        # Scheefheid: 0 is symmetrisch, > 1 is sterk scheef naar rechts
        stats["scheefheid"] = self.df[cols].skew()
        return stats.round(2)

    def categorical_stats(self):
        """Aantal en percentage per categorie van iedere tekstkolom."""
        tables = []
        for col in self.categorical_cols:
            counts = self.df[col].value_counts()
            table = pd.DataFrame({
                "kolom": col,
                "categorie": counts.index,
                "aantal": counts.values,
                "percentage": (counts.values / len(self.df) * 100).round(2),
            })
            tables.append(table)
        return pd.concat(tables, ignore_index=True)

    def duplicates(self):
        """Telt dubbele rijen, zonder naar het unieke id te kijken."""
        return self.df.drop(columns=self.id_col).duplicated().sum()


eda = StrokeEDA(data.train)
print("Numerieke kolommen:  ", eda.numeric_cols)
print("Binaire kolommen:    ", eda.binary_cols)
print("Categorische kolommen:", eda.categorical_cols)
print("Dubbele rijen:       ", eda.duplicates())
eda.overview()

In [ ]:
# Basisstatistieken van de getallen
eda.numeric_stats()

In [ ]:
# Verdeling van de categorieën per tekstkolom
eda.categorical_stats()

In [ ]:
# Ontbrekende waarden in train en test naast elkaar
pd.DataFrame({
    "train": data.train.isna().sum(),
    "test": data.test.isna().sum(),
})

**Wat valt op?** *(controleer de getallen na het runnen)*

* `bmi` is de enige kolom met ontbrekende waarden, in zowel train als test. Die moeten we opvullen of weglaten, want scikit-learn kan niet met lege waarden rekenen (zie 2.4).
* `age`, `avg_glucose_level` en `bmi` hebben heel verschillende schalen (tientallen tegenover honderden). Dat is belangrijk voor standaardisatie (zie 2.8).
* `avg_glucose_level` is scheef naar rechts: de meeste mensen zitten rond de 80 tot 110, maar een groep heeft veel hogere waarden (mogelijk diabetes).
* `gender` heeft een zeer kleine categorie `Other`. Daar zijn te weinig rijen van om iets uit te leren.

### 2.4 Voldoet de data aan de eisen van Scikit-Learn?

Scikit-learn stelt een paar minimale eisen aan de data voordat je een model kunt trainen (scikit-learn developers, z.d.):

1. **Alles is numeriek.** Modellen rekenen met getallen; tekst zoals `"Male"` of `"Urban"` moet eerst worden omgezet, bijvoorbeeld met one-hot encoding (dummy-kolommen).
2. **Geen ontbrekende waarden.** De meeste modellen (KNN, SVM, logistische regressie) geven een foutmelding bij `NaN`.
3. **Geen oneindige waarden** (`inf`).
4. **De juiste vorm.** X is een tabel (2D: rijen × features) en y is één kolom (1D) met evenveel rijen als X.
5. **Train en test hebben dezelfde features** in dezelfde volgorde, anders kan het getrainde model de testdata niet voorspellen.

De functie hieronder controleert deze eisen automatisch.

In [ ]:
def check_sklearn_ready(X, y=None):
    """Controleert of X (en y) voldoen aan de minimale eisen van sklearn."""
    non_numeric = list(X.select_dtypes(exclude="number").columns)
    numeric = X.select_dtypes(include="number")
    checks = {
        "alle features numeriek": len(non_numeric) == 0,
        "geen ontbrekende waarden": X.isna().sum().sum() == 0,
        "geen oneindige waarden": not np.isinf(numeric.to_numpy()).any(),
        "X is 2D": X.ndim == 2,
    }
    if y is not None:
        checks["y is 1D"] = y.ndim == 1
        checks["X en y even lang"] = len(X) == len(y)

    for check, ok in checks.items():
        print(f"{'OK  ' if ok else 'FOUT'}  {check}")
    if non_numeric:
        print(f"\nNiet-numerieke kolommen: {non_numeric}")
    missing = X.isna().sum()
    if missing.any():
        print("Kolommen met ontbrekende waarden:",
              missing[missing > 0].to_dict())
    return all(checks.values())


X_raw = data.train.drop(columns=["id", "stroke"])
y_raw = data.train["stroke"]
check_sklearn_ready(X_raw, y_raw)

De ruwe data voldoet **nog niet** aan de eisen:

* De tekstkolommen moeten numeriek worden. We gebruiken **one-hot encoding**: voor iedere categorie komt een kolom met 0 of 1. We kiezen hiervoor en niet voor nummeren (bijvoorbeeld `Private` = 1, `Self-employed` = 2), omdat nummeren een volgorde suggereert die er niet is. KNN en SVM zouden dan denken dat `Govt_job` "dichter bij" `Private` ligt dan `children`.
* De ontbrekende `bmi`-waarden moeten worden opgevuld.

Deze aanpassingen voeren we uit in 2.7.

### 2.5 Relaties tussen de variabelen en stroke

We onderzoeken nu welke features samenhangen met `stroke`. We beginnen met de verdeling van de doelvariabele zelf, daarna bekijken we per soort kolom de relatie met een passende grafiek:

| Soort feature | Grafiek | Waarom |
|---|---|---|
| doelvariabele | staafdiagram | laat zien hoe scheef de klassen verdeeld zijn |
| numeriek | histogram en boxplot per klasse | laat zien of de verdeling verschuift bij stroke, en toont outliers |
| binair en categorisch | percentage stroke per categorie | eerlijker dan aantallen, omdat categorieën verschillend groot zijn |
| alles samen | correlatiematrix | geeft in één oogopslag de sterkte van alle lineaire verbanden |

De plotfuncties voegen we toe aan de class `StrokeEDA`, zodat alle EDA-code bij elkaar blijft.

In [ ]:
def plot_target_balance(self):
    """Staafdiagram van het aantal patiënten met en zonder stroke."""
    counts = self.df[self.target].value_counts().sort_index()
    percentages = counts / counts.sum() * 100

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Geen stroke (0)", "Stroke (1)"], counts.values,
           color=["tab:blue", "tab:red"])
    for i, (n, pct) in enumerate(zip(counts.values, percentages.values)):
        ax.text(i, n, f"{n} ({pct:.1f}%)", ha="center", va="bottom")
    ax.set_title("Verdeling van de doelvariabele stroke")
    ax.set_ylabel("Aantal patiënten")
    plt.show()
    return percentages.round(2)


StrokeEDA.plot_target_balance = plot_target_balance
eda.plot_target_balance()

De data is sterk **imbalanced**: maar een klein percentage van de patiënten heeft een stroke gehad. Dat is logisch, want in de echte bevolking krijgt ook maar een klein deel van de mensen een herseninfarct. Gevolg: een model dat altijd "geen stroke" zegt, haalt al een accuracy van boven de 90% maar vindt geen enkele patiënt. Daarom gebruiken we in hoofdstuk 3 de F1-score en in hoofdstuk 4 technieken voor imbalanced data.

In [ ]:
def plot_numeric_vs_target(self):
    """Histogram en boxplot van iedere numerieke kolom, per klasse."""
    n_cols = len(self.numeric_cols)
    fig, axes = plt.subplots(n_cols, 2, figsize=(12, 4 * n_cols))
    for row, col in enumerate(self.numeric_cols):
        # Histogram: common_norm=False zodat beide klassen even hoog worden
        # getekend, anders verdwijnt de kleine stroke-groep in de grafiek
        sns.histplot(data=self.df, x=col, hue=self.target, stat="density",
                     common_norm=False, bins=40, kde=True,
                     ax=axes[row, 0])
        axes[row, 0].set_title(f"Verdeling van {col} per klasse")

        # Boxplot: mediaan, spreiding en outliers per klasse
        sns.boxplot(data=self.df, x=self.target, y=col, ax=axes[row, 1])
        axes[row, 1].set_title(f"Boxplot van {col} per klasse")
        axes[row, 1].set_xticks([0, 1], ["Geen stroke", "Stroke"])
    plt.tight_layout()
    plt.show()


def numeric_means_by_target(self):
    """Gemiddelde en mediaan van de numerieke kolommen per klasse."""
    return (self.df.groupby(self.target)[self.numeric_cols]
            .agg(["mean", "median"]).round(1).T)


StrokeEDA.plot_numeric_vs_target = plot_numeric_vs_target
StrokeEDA.numeric_means_by_target = numeric_means_by_target
eda.plot_numeric_vs_target()
eda.numeric_means_by_target()

In [ ]:
def outliers_iqr(self):
    """Telt per numerieke kolom de outliers volgens de IQR-methode.

    Een waarde is een outlier als die meer dan 1,5 keer de
    interkwartielafstand (IQR = Q3 - Q1) onder Q1 of boven Q3 ligt.
    """
    rows = []
    for col in self.numeric_cols:
        q1, q3 = self.df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        is_outlier = (self.df[col] < low) | (self.df[col] > high)
        rows.append({
            "kolom": col,
            "ondergrens": round(low, 1),
            "bovengrens": round(high, 1),
            "min": self.df[col].min(),
            "max": self.df[col].max(),
            "outliers": is_outlier.sum(),
            "outliers_%": round(is_outlier.mean() * 100, 2),
            # Hebben de outliers vaker stroke dan gemiddeld?
            "stroke_%_outliers": round(
                self.df.loc[is_outlier, self.target].mean() * 100, 2),
        })
    return pd.DataFrame(rows)


StrokeEDA.outliers_iqr = outliers_iqr
print(f"Stroke-percentage in de hele trainingsdata: "
      f"{eda.df['stroke'].mean() * 100:.2f}%")
eda.outliers_iqr()

**Outliers.** Een outlier volgens de IQR-regel is niet automatisch een fout. Een glucosewaarde van 250 mg/dL is hoog, maar komt echt voor bij mensen met diabetes. Als het stroke-percentage bij de outliers hoger is dan gemiddeld, bevatten ze juist nuttige informatie en verwijderen we ze **niet**. We verwijderen alleen waarden die medisch onmogelijk zijn (zie 2.7).

In [ ]:
def plot_stroke_rate(self, cols=None):
    """Percentage stroke per categorie voor binaire en tekstkolommen."""
    if cols is None:
        cols = self.binary_cols + self.categorical_cols
    overall = self.df[self.target].mean() * 100

    n_plots = len(cols)
    n_rows = int(np.ceil(n_plots / 3))
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    for ax, col in zip(axes, cols):
        rate = self.df.groupby(col)[self.target].agg(["mean", "size"])
        rate["mean"] *= 100
        ax.bar(rate.index.astype(str), rate["mean"])
        # Aantal patiënten per categorie boven iedere staaf
        for i, (pct, n) in enumerate(zip(rate["mean"], rate["size"])):
            ax.text(i, pct, f"n={n}", ha="center", va="bottom", fontsize=8)
        ax.axhline(overall, color="red", linestyle="--",
                   label=f"gemiddeld ({overall:.1f}%)")
        ax.set_title(f"Stroke-percentage per {col}")
        ax.set_ylabel("% stroke")
        ax.tick_params(axis="x", rotation=30)
        ax.legend(fontsize=8)
    # Lege subplots weghalen
    for ax in axes[n_plots:]:
        ax.remove()
    plt.tight_layout()
    plt.show()


StrokeEDA.plot_stroke_rate = plot_stroke_rate
eda.plot_stroke_rate()

De rode stippellijn is het gemiddelde stroke-percentage. Staven die er ver boven liggen, horen bij groepen met een verhoogd risico. Let op de `n=` boven de staven: een hoog percentage in een kleine groep (zoals `gender = Other` of `Never_worked`) is weinig betrouwbaar.

Sommige categorieën hangen vooral samen met **leeftijd**. Kinderen (`work_type = children`) zijn bijna nooit getrouwd en roken vaak "Unknown". Als `ever_married = Yes` een hoger risico laat zien, komt dat waarschijnlijk doordat getrouwde mensen gemiddeld ouder zijn, niet door het huwelijk zelf. De grafiek hieronder controleert dat.

In [ ]:
def plot_stroke_rate_by_age(self, by_col, bins=(0, 20, 40, 60, 70, 80, 120)):
    """Stroke-percentage per leeftijdsgroep, apart voor iedere categorie."""
    age_group = pd.cut(self.df["age"], bins=bins, right=False)
    rate = (self.df.groupby([age_group, by_col], observed=True)[self.target]
            .mean().mul(100).unstack())

    rate.plot(kind="bar", figsize=(10, 4))
    plt.title(f"Stroke-percentage per leeftijdsgroep en {by_col}")
    plt.xlabel("Leeftijdsgroep (jaren)")
    plt.ylabel("% stroke")
    plt.xticks(rotation=0)
    plt.legend(title=by_col)
    plt.show()


StrokeEDA.plot_stroke_rate_by_age = plot_stroke_rate_by_age
eda.plot_stroke_rate_by_age("ever_married")
eda.plot_stroke_rate_by_age("hypertension")

In [ ]:
def plot_age_glucose(self, sample_size=5000, random_state=42):
    """Scatterplot van leeftijd tegen glucose, gekleurd op stroke."""
    # Een steekproef houdt de grafiek leesbaar; alle stroke-gevallen
    # blijven erin zodat de kleine klasse goed zichtbaar is
    stroke = self.df[self.df[self.target] == 1]
    no_stroke = self.df[self.df[self.target] == 0].sample(
        n=min(sample_size, (self.df[self.target] == 0).sum()),
        random_state=random_state)
    plt.figure(figsize=(9, 5))
    plt.scatter(no_stroke["age"], no_stroke["avg_glucose_level"],
                s=6, alpha=0.3, label="Geen stroke (steekproef)")
    plt.scatter(stroke["age"], stroke["avg_glucose_level"],
                s=10, alpha=0.7, color="tab:red", label="Stroke")
    plt.title("Leeftijd tegen gemiddeld glucosegehalte")
    plt.xlabel("age")
    plt.ylabel("avg_glucose_level")
    plt.legend()
    plt.show()


StrokeEDA.plot_age_glucose = plot_age_glucose
eda.plot_age_glucose()

**Correlaties.** Voor de correlatiematrix zetten we de tekstkolommen tijdelijk om naar dummy-kolommen. De correlatie van een 0/1-kolom met `stroke` is hetzelfde als de *point-biserial* correlatie, dus dat mag. Let op: correlatie meet alleen **lineaire** verbanden. Een feature met een lage correlatie kan voor een decision tree nog steeds nuttig zijn.

In [ ]:
def correlations(self):
    """Correlatiematrix van alle features na tijdelijke one-hot encoding."""
    encoded = pd.get_dummies(self.df.drop(columns=self.id_col),
                             columns=self.categorical_cols, dtype=int)
    return encoded.corr()


def plot_correlations(self):
    """Heatmap van alle correlaties en een staafdiagram voor stroke."""
    corr = self.correlations()

    plt.figure(figsize=(13, 10))
    # Alleen de onderste driehoek tonen, de bovenste is een spiegelbeeld
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
                annot=True, fmt=".2f", annot_kws={"size": 6})
    plt.title("Correlatiematrix (na one-hot encoding)")
    plt.show()

    target_corr = (corr[self.target].drop(self.target)
                   .sort_values(key=abs, ascending=False))
    plt.figure(figsize=(8, 7))
    sns.barplot(x=target_corr.values, y=target_corr.index,
                hue=target_corr.values > 0, palette=["tab:blue", "tab:red"],
                legend=False)
    plt.title("Correlatie van iedere feature met stroke")
    plt.xlabel("Correlatiecoëfficiënt")
    plt.ylabel("")
    plt.show()
    return target_corr.round(3)


StrokeEDA.correlations = correlations
StrokeEDA.plot_correlations = plot_correlations
eda.plot_correlations()

### 2.6 Belangrijkste bevindingen

> **Let op voor de groep:** onderstaande tekst is een opzet. Vul de getallen in en pas de conclusies aan op basis van de uitkomsten die jullie zien na het runnen van de cellen hierboven.

1. **Sterk imbalanced.** Slechts ongeveer \_\_% van de patiënten heeft een stroke gehad. Accuracy is daardoor misleidend; we gebruiken de F1-score en passen technieken voor imbalanced data toe (hoofdstuk 4).
2. **Leeftijd is de sterkste voorspeller.** Patiënten met een stroke zijn gemiddeld \_\_ jaar oud, tegenover \_\_ jaar zonder stroke. Onder de \_\_ jaar komt stroke vrijwel niet voor. De correlatie met stroke is \_\_.
3. **Hoge bloeddruk, hartziekte en glucose verhogen het risico.** Bij `hypertension = 1` is het stroke-percentage \_\_% tegenover \_\_%; bij `heart_disease = 1` \_\_%. Patiënten met een stroke hebben vaker een hoge `avg_glucose_level`. Dit past bij de medische literatuur (O'Donnell et al., 2016).
4. **Sommige verbanden lopen via leeftijd.** `ever_married`, `work_type = children` en `smoking_status = Unknown` hangen sterk samen met leeftijd. Hun verband met stroke is daardoor deels schijn (confounding). We houden ze wel in de data, omdat ze voor het model informatie kunnen bevatten.
5. **Weinig effect van woonomgeving en geslacht.** `Residence_type` en `gender` laten nauwelijks verschil zien in stroke-percentage.
6. **BMI heeft ontbrekende waarden en zwak verband.** \_\_% van `bmi` ontbreekt. Het verband met stroke is zwakker dan dat van leeftijd en glucose.
7. **Outliers zijn echt.** De outliers in `avg_glucose_level` en `bmi` zijn medisch mogelijk en hebben een hoger stroke-percentage dan gemiddeld. We behouden ze.
8. **Verschillende schalen.** `age`, `avg_glucose_level` en `bmi` hebben heel verschillende bereiken, dus standaardisatie is nodig voor KNN, SVM en logistische regressie (2.8).

### 2.7 Data opschonen

Op basis van 2.3 t/m 2.6 maken we de volgende keuzes:

| Stap | Wat | Waarom |
|---|---|---|
| 1 | `id` niet als feature gebruiken | Een volgnummer heeft niets met gezondheid te maken. Een model zou er toevallige patronen in kunnen vinden (overfitting). We bewaren `id` wel apart voor de Kaggle-inlevering. |
| 2 | Dubbele rijen uit de trainingsdata verwijderen | Dubbele rijen tellen dubbel mee bij het trainen en maken de cross-validatie te optimistisch, omdat dezelfde patiënt dan in train- én validatiedeel kan zitten. |
| 3 | `gender = Other` vervangen door de meest voorkomende waarde | Deze groep is zo klein dat een model er niets uit kan leren. Rijen weghalen kan niet in `test.csv` (Kaggle wil voor ieder `id` een voorspelling), dus vervangen we de waarde in train en test op dezelfde manier. |
| 4 | Onmogelijke waarden leegmaken | Een `bmi` onder de 10 of boven de 100 is medisch niet realistisch en waarschijnlijk een invoerfout. We behandelen die als ontbrekend. |
| 5 | Ontbrekende `bmi` opvullen met de **mediaan van train** | De mediaan is ongevoelig voor outliers (beter dan het gemiddelde bij een scheve verdeling). We berekenen hem alleen op train en gebruiken dezelfde waarde voor test, zodat er geen informatie uit de testdata "lekt". |
| 6 | One-hot encoding van de tekstkolommen | Maakt de data numeriek zonder een nep-volgorde (zie 2.4). Met `drop_first=True` laten we per kolom één categorie weg: die is af te leiden uit de andere (alle nullen), en zo voorkomen we perfecte multicollineariteit bij logistische regressie. |
| 7 | Test dezelfde kolommen geven als train | Als een categorie in test niet voorkomt, ontbreekt die dummy-kolom. We vullen hem dan met 0, zodat het model precies dezelfde features krijgt. |

We behouden de outliers en `smoking_status = Unknown` als eigen categorie, omdat "onbekend" zelf informatie bevat (het komt vooral bij kinderen voor).

Net als een scikit-learn transformer heeft de class `StrokeCleaner` een `fit` (leren van train) en een `transform` (toepassen op train én test).

In [ ]:
class StrokeCleaner:
    """Schoont de stroke-data op en zet tekstkolommen om naar dummy's.

    fit() leert de benodigde waarden (mediaan, meest voorkomende
    categorie, kolomnamen) alleen van de trainingsdata. transform()
    past precies dezelfde stappen toe op train en test.
    """

    def __init__(self, id_col="id", target="stroke",
                 bmi_range=(10, 100)):
        self.id_col = id_col
        self.target = target
        self.bmi_range = bmi_range
        self.bmi_median = None
        self.gender_mode = None
        self.categorical_cols = None
        self.feature_names = None

    def _fix_values(self, df):
        """Stap 3 en 4: gender Other vervangen en onmogelijke bmi leegmaken."""
        df = df.copy()
        if "gender" in df.columns and self.gender_mode is not None:
            df["gender"] = df["gender"].replace("Other", self.gender_mode)
        if "bmi" in df.columns:
            low, high = self.bmi_range
            impossible = (df["bmi"] < low) | (df["bmi"] > high)
            df.loc[impossible, "bmi"] = np.nan
        return df

    def fit(self, df):
        """Leert de opschoonwaarden van de trainingsdata."""
        if "gender" in df.columns:
            self.gender_mode = df.loc[df["gender"] != "Other",
                                      "gender"].mode()[0]
        df = self._fix_values(df)
        self.bmi_median = df["bmi"].median()

        features = df.drop(columns=[self.id_col, self.target],
                           errors="ignore")
        self.categorical_cols = list(
            features.select_dtypes(exclude="number").columns)
        # Kolomnamen na encoding onthouden om test gelijk te trekken
        self.feature_names = list(self._encode(features).columns)
        return self

    def _encode(self, features):
        """Stap 6: one-hot encoding van de tekstkolommen."""
        return pd.get_dummies(features, columns=self.categorical_cols,
                              drop_first=True, dtype=int)

    def transform(self, df):
        """Geeft (ids, X, y) terug; y is None als stroke ontbreekt."""
        df = self._fix_values(df)
        df["bmi"] = df["bmi"].fillna(self.bmi_median)

        ids = df[self.id_col]
        y = df[self.target] if self.target in df.columns else None
        features = df.drop(columns=[self.id_col, self.target],
                           errors="ignore")
        # Stap 7: zelfde kolommen als train, ontbrekende dummy's worden 0
        X = self._encode(features).reindex(columns=self.feature_names,
                                           fill_value=0)
        return ids, X, y


# Stap 2: dubbele rijen (zonder te kijken naar id) alleen uit train halen
feature_and_target = data.train.columns.drop("id")
train_clean = data.train.drop_duplicates(subset=feature_and_target)
print(f"Verwijderde dubbele rijen: {len(data.train) - len(train_clean)}")

cleaner = StrokeCleaner().fit(train_clean)
train_ids, X_train, y_train = cleaner.transform(train_clean)
test_ids, X_test, _ = cleaner.transform(data.test)

print(f"Mediaan bmi (train): {cleaner.bmi_median}")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, "
      f"X_test: {X_test.shape}\n")
check_sklearn_ready(X_train, y_train)
print()
check_sklearn_ready(X_test)
X_train.head()

Na het opschonen voldoen zowel `X_train` als `X_test` aan alle eisen van scikit-learn, en hebben ze precies dezelfde kolommen.

### 2.8 Standaardiseren

**Wat is standaardiseren?** Bij standaardiseren zet je iedere numerieke kolom om naar dezelfde schaal, zodat hij een gemiddelde van 0 en een standaardafwijking van 1 heeft. Voor iedere waarde $x$ in een kolom bereken je de *z-score*:

$$z = \frac{x - \mu}{\sigma}$$

met $\mu$ het gemiddelde en $\sigma$ de standaardafwijking van die kolom in de trainingsdata. Een $z$ van 2 betekent: twee standaardafwijkingen boven het gemiddelde. De vorm van de verdeling verandert niet, alleen de schaal (scikit-learn developers, z.d.).

**Wanneer is het gewenst?** Als features een heel verschillende schaal hebben én het model gevoelig is voor die schaal:

* **KNN** en **SVM** rekenen met afstanden. Zonder standaardisatie bepaalt `avg_glucose_level` (bereik ca. 55 tot 270) de afstand bijna helemaal, en telt `age` of een 0/1-kolom nauwelijks mee.
* **Logistische regressie** met regularisatie straft grote coëfficiënten. Zonder gelijke schaal wordt de ene feature zwaarder gestraft dan de andere. Ook convergeert gradient descent sneller op gestandaardiseerde data.
* **Decision trees en de ensembles daarvan** (Random Forest, Gradient Boosting, XGBoost) hebben het **niet** nodig. Een boom splitst op één feature tegelijk ("is `age` > 60?"), en de volgorde van de waarden blijft bij standaardiseren hetzelfde.

**Voegt het waarde toe voor deze dataset?** Ja. `age`, `avg_glucose_level` en `bmi` hebben heel verschillende bereiken (zie 2.3), en we gebruiken KNN, SVM en logistische regressie. We standaardiseren alleen deze drie numerieke kolommen. De dummy- en binaire kolommen laten we 0/1, zodat ze makkelijk te interpreteren blijven.

**Belangrijk:** de scaler leert $\mu$ en $\sigma$ alleen van de trainingsdata. Bij cross-validatie in hoofdstuk 4 zetten we de scaler daarom in een `Pipeline`, zodat hij per fold opnieuw alleen op het trainingsdeel wordt gefit. Anders lekt informatie van het validatiedeel naar het model en is de score te optimistisch. Hieronder laten we zien wat standaardiseren doet.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NUMERIC_COLS = ["age", "avg_glucose_level", "bmi"]


def make_scaler(numeric_cols=NUMERIC_COLS):
    """Scaler die alleen de numerieke kolommen standaardiseert.

    De overige (0/1) kolommen gaan ongewijzigd door. Deze functie
    gebruiken we straks in de Pipelines van KNN, SVM en LogReg.
    """
    return ColumnTransformer(
        [("num", StandardScaler(), numeric_cols)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


scaler = make_scaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)[X_train.columns]
X_test_scaled = scaler.transform(X_test)[X_train.columns]

# Voor en na: gemiddelde en standaardafwijking van de numerieke kolommen
pd.DataFrame({
    "gemiddelde_voor": X_train[NUMERIC_COLS].mean(),
    "std_voor": X_train[NUMERIC_COLS].std(),
    "gemiddelde_na": X_train_scaled[NUMERIC_COLS].mean(),
    "std_na": X_train_scaled[NUMERIC_COLS].std(),
}).round(2)

In [ ]:
def plot_before_after(before, after, cols):
    """Boxplots van de numerieke kolommen voor en na standaardiseren."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(data=before[cols], ax=axes[0])
    axes[0].set_title("Voor standaardiseren")
    sns.boxplot(data=after[cols], ax=axes[1])
    axes[1].set_title("Na standaardiseren (z-scores)")
    plt.tight_layout()
    plt.show()


plot_before_after(X_train, X_train_scaled, NUMERIC_COLS)

Voor het standaardiseren domineert `avg_glucose_level` de schaal. Daarna liggen alle drie de kolommen rond 0 met een vergelijkbare spreiding, en de vorm van iedere verdeling (inclusief de outliers) is gelijk gebleven.

Voor de modellen in hoofdstuk 4 gebruiken we `X_train`, `y_train` en `X_test` uit 2.7, en zetten we `make_scaler()` in een `Pipeline` bij de modellen die gevoelig zijn voor schaal. `X_train_scaled` hierboven dient alleen om het effect te laten zien.

## 3. Evaluatie

### 3.1 Metrics

### 3.2 Confusion matrix

### 3.3 Waarom F1-score?

## 4. Modellen

### 4.1 Imbalanced data

### 4.2 K-Nearest Neighbors

#### Uitleg
#### Resultaten

### 4.3 Logistic Regression

#### Uitleg
#### Resultaten

### 4.4 Support Vector Machine

#### Uitleg
#### Resultaten

### 4.5 Decision Tree

#### Uitleg
#### Resultaten

### 4.6 Random Forest

#### Uitleg
#### Resultaten

### 4.7 Gradient Boosting

#### Uitleg
#### Resultaten

### 4.8 XGBoost

#### Uitleg
#### Resultaten

### 4.9 Eigen ensemble

#### Uitleg
#### Resultaten

## 5. Conclusie en advies

### 5.1 Overzicht scores

| Model | F1 notebook | F1 Kaggle |
|---|---|---|
| KNN | | |
| Logistic Regression | | |
| SVM | | |
| Decision Tree | | |
| Random Forest | | |
| Gradient Boosting | | |
| XGBoost | | |
| Eigen ensemble | | |

### 5.2 Welk model presteert het beste?

### 5.3 Andere overwegingen

### 5.4 Advies aan artsen

## 6. Referenties

O'Donnell, M. J., Chin, S. L., Rangarajan, S., Xavier, D., Liu, L., Zhang, H., ... Yusuf, S. (2016). Global and regional effects of potentially modifiable risk factors associated with acute stroke in 32 countries (INTERSTROKE): A case-control study. *The Lancet, 388*(10046), 761–775. https://doi.org/10.1016/S0140-6736(16)30506-2

scikit-learn developers. (z.d.). *Preprocessing data*. scikit-learn. Geraadpleegd op 10 oktober 2026, van https://scikit-learn.org/stable/modules/preprocessing.html